<a href="https://colab.research.google.com/github/syrine1O/ml-finance-project/blob/T11-lightgbm/notebooks/05_lightgbm_T11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [18]:
import os
import sys
from itertools import product
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.model_selection import TimeSeriesSplit


here = Path.cwd().resolve()
root = next(
    (folder for folder in (here, *here.parents)
     if (folder / "src" / "utils.py").is_file()),
    None,
)

if root is None:
    raise FileNotFoundError(
        "Le dépôt ml-finance-project est introuvable. "
        "Placez la session dans le dossier du dépôt ou dans un de ses sous-dossiers."
    )

os.chdir(root)
sys.path.insert(0, str(root))
from src.utils import TEST_START, qlike, save_forecasts

HORIZONS = (1, 5, 22)
X_COLS = ["rv_d", "rv_w", "rv_m", "ret", "ret_neg", "vix"]
PARAMS = dict(objective="gamma", subsample=0.8, subsample_freq=1,
              colsample_bytree=0.8, random_state=42, verbose=-1, n_jobs=2)
GRID = {
    "num_leaves": [7, 15, 31],
    "learning_rate": [0.01, 0.03],
    "min_child_samples": [50, 200],
    "n_estimators": [300, 800],
}
VARIANTS = {"LGBM": X_COLS, "LGBM_noVIX": [col for col in X_COLS if col != "vix"]}
TUNING_START = pd.Timestamp("2000-01-01")
TUNING_END = min(pd.Timestamp(TEST_START), pd.Timestamp("2015-01-01"))
results_dir = root / "results"
results_dir.mkdir(exist_ok=True)

features = {}
for ticker in ("spx", "tsx"):
    path = root / "data" / f"features_{ticker}.csv"
    frame = pd.read_csv(path, index_col=0, parse_dates=True).sort_index()
    required = X_COLS + [f"y_h{h}" for h in HORIZONS]
    missing = set(required) - set(frame.columns)
    if missing or not isinstance(frame.index, pd.DatetimeIndex) or not frame.index.is_unique:
        raise ValueError(f"{path.name} : dates invalides ou colonnes manquantes {sorted(missing)}")
    for h in HORIZONS:
        target = frame[f"y_h{h}"].dropna()
        if not np.isfinite(target).all() or (target <= 0).any():
            raise ValueError(f"{path.name} : les cibles observées y_h{h} doivent être finies et positives.")
    features[ticker] = frame
    print(ticker, len(frame), frame.index.min().date(), frame.index.max().date())

spx 6690 2000-02-03 2026-09-10
tsx 6670 2000-02-03 2026-09-09


Sélection des hyperparamètres sur 2000–2014

La validation croisée comprend quatre plis chronologiques. Un écart de 22 observations sépare l’entraînement de la validation. Les observations dont la cible utilise des données de 2015 sont exclues du réglage. Pour chaque indice et chaque horizon, les variantes avec et sans VIX sont réglées séparément sur la même grille. La combinaison retenue minimise la QLIKE moyenne de validation.

In [19]:
def tune(frame, h, x_cols):
    target = f"y_h{h}"
    # Calculer la fin de fenêtre avant de filtrer les dates ou les cibles.
    target_end = frame.index.to_series().shift(-h)
    train = frame.loc[
        (frame.index >= TUNING_START) & (target_end < TUNING_END)
    ].dropna(subset=[target])
    x, y = train[x_cols], train[target]
    folds = list(TimeSeriesSplit(n_splits=4, gap=22).split(train))
    comparisons = []
    for values in product(*GRID.values()):
        candidate = dict(zip(GRID, values))
        scores = []
        for fit_idx, val_idx in folds:
            model = lgb.LGBMRegressor(**(PARAMS | candidate))
            model.fit(x.iloc[fit_idx], y.iloc[fit_idx])
            pred = np.maximum(model.predict(x.iloc[val_idx]), 1e-4)
            scores.append(float(qlike(y.iloc[val_idx].to_numpy(), pred)))
        comparisons.append(candidate | {"QLIKE_cv": float(np.mean(scores))})
    ranking = pd.DataFrame(comparisons)
    if not np.isfinite(ranking["QLIKE_cv"]).all():
        raise ValueError(f"QLIKE de validation invalide à h={h}.")
    return ranking.sort_values("QLIKE_cv", kind="stable").reset_index(drop=True)

selected = {}
summary = []
for ticker, frame in features.items():
    for name, x_cols in VARIANTS.items():
        for h in HORIZONS:
            print(f"Validation : {ticker} | {name} | h={h}", flush=True)
            ranking = tune(frame, h, x_cols)
            display(ranking.head(5))
            best = ranking.iloc[0]
            grid_params = {key: type(grid[0])(best[key]) for key, grid in GRID.items()}
            selected[ticker, name, h] = PARAMS | grid_params
            summary.append({
                "index": ticker, "model": name, "horizon": h,
                **grid_params, "QLIKE_cv": best["QLIKE_cv"],
            })

params_table = pd.DataFrame(summary)
params_table.to_csv(results_dir / "table5_lgbm_params.csv", index=False)
display(params_table)

Validation : spx | LGBM | h=1


,num_leaves,learning_rate,min_child_samples,n_estimators,QLIKE_cv
0,7,0.01,50,800,0.353438
1,7,0.03,50,300,0.354649
2,15,0.01,50,800,0.360487
3,7,0.03,50,800,0.360497
4,15,0.03,50,300,0.362600


Validation : spx | LGBM | h=5


,num_leaves,learning_rate,min_child_samples,n_estimators,QLIKE_cv
0,7,0.01,50,800,0.201125
1,15,0.03,50,300,0.201242
2,7,0.03,50,300,0.202632
3,15,0.01,50,800,0.202828
4,31,0.03,50,300,0.203894


Validation : spx | LGBM | h=22


,num_leaves,learning_rate,min_child_samples,n_estimators,QLIKE_cv
0,15,0.01,50,800,0.254069
1,7,0.01,50,800,0.254238
2,15,0.01,50,300,0.254933
3,31,0.01,50,300,0.255851
4,7,0.03,50,300,0.255935


... (15 outputs left out)

,num_leaves,learning_rate,min_child_samples,n_estimators,QLIKE_cv
0,7,0.01,50,800,0.316833
1,7,0.03,50,300,0.319771
2,15,0.01,50,800,0.322005
3,7,0.01,200,800,0.325074
4,15,0.03,50,300,0.325260


Validation : tsx | LGBM_noVIX | h=22


,num_leaves,learning_rate,min_child_samples,n_estimators,QLIKE_cv
0,15,0.03,200,300,0.366793
1,31,0.03,200,300,0.366793
2,7,0.03,200,300,0.366850
3,7,0.03,200,800,0.366863
4,15,0.03,200,800,0.366999


,index,model,horizon,num_leaves,learning_rate,min_child_samples,n_estimators,QLIKE_cv
0,spx,LGBM,1,7,0.01,50,800,0.353438
1,spx,LGBM,5,7,0.01,50,800,0.201125
2,spx,LGBM,22,15,0.01,50,800,0.254069
3,spx,LGBM_noVIX,1,7,0.03,50,300,0.383574
4,spx,LGBM_noVIX,5,15,0.03,50,300,0.218360
5,spx,LGBM_noVIX,22,7,0.01,50,800,0.275507
6,tsx,LGBM,1,7,0.03,50,300,0.432744
7,tsx,LGBM,5,7,0.03,50,300,0.310916
8,tsx,LGBM,22,15,0.03,200,300,0.373140
9,tsx,LGBM_noVIX,1,7,0.01,50,800,0.441347


Prévisions hors échantillon

Les hyperparamètres sont fixés avant l’évaluation. Le modèle est réentraîné toutes les 66 séances à partir des cibles déjà observées à la date de prévision. Les dates dépourvues de variance réalisée à l’horizon considéré sont exclues des fichiers d’évaluation.

In [20]:
def lgb_walk_forward(frame, h, params, x_cols, name, refit=66):
    target = f"y_h{h}"
    test_dates = frame.index[(frame.index >= TEST_START) & frame[target].notna()]
    rows = []
    for step, date in enumerate(test_dates):
        if step % refit == 0:
            train = frame.loc[:date].iloc[:-h].dropna(subset=[target])
            if train.empty:
                raise ValueError(f"Historique insuffisant au {date.date()} pour h={h}.")
            model = lgb.LGBMRegressor(**params).fit(train[x_cols], train[target])
        forecast = model.predict(frame.loc[[date], x_cols])[0]
        rows.append({
            "date": date, "model": name, "horizon": h,
            "forecast": max(float(forecast), 1e-4), "actual": frame.at[date, target],
        })
    return pd.DataFrame(rows, columns=["date", "model", "horizon", "forecast", "actual"])

exports = {}
for ticker, frame in features.items():
    for name, x_cols in VARIANTS.items():
        forecasts = pd.concat([
            lgb_walk_forward(frame, h, selected[ticker, name, h], x_cols, name)
            for h in HORIZONS
        ], ignore_index=True)
        if (set(forecasts["horizon"]) != set(HORIZONS)
                or forecasts.duplicated(["date", "horizon"]).any()
                or not np.isfinite(forecasts[["forecast", "actual"]]).all().all()
                or (forecasts[["forecast", "actual"]] <= 0).any().any()):
            raise ValueError(f"Prévisions invalides : {ticker} / {name}")
        exports[ticker, name] = forecasts
        print(ticker, name, forecasts.groupby("horizon").size().to_dict())

for ticker in features:
    with_vix = exports[ticker, "LGBM"][["date", "horizon", "actual"]]
    no_vix = exports[ticker, "LGBM_noVIX"][["date", "horizon", "actual"]]
    if not with_vix.equals(no_vix):
        raise ValueError(f"Dates ou cibles différentes entre les variantes pour {ticker}.")

for (ticker, name), forecasts in exports.items():
    save_forecasts(forecasts, name, ticker)

spx LGBM {1: 2939, 5: 2939, 22: 2939}
spx LGBM_noVIX {1: 2939, 5: 2939, 22: 2939}
tsx LGBM {1: 2933, 5: 2933, 22: 2933}
tsx LGBM_noVIX {1: 2933, 5: 2933, 22: 2933}


Comparaison des modèles avec et sans VIX

La QLIKE mesure l’écart entre la variance prévue et la variance observée. Une valeur inférieure indique une meilleure performance selon ce critère. La comparaison hors échantillon repose sur les mêmes dates, horizons et valeurs observées pour les deux variantes. Comme les hyperparamètres sont sélectionnés séparément, l’écart de QLIKE compare les performances des deux configurations retenues.


In [21]:
from src.utils import evaluate

print(evaluate("spx"))
print(evaluate("tsx"))

    horizon       model   QLIKE     MSE  DM vs HAR  p-value
0         1      EGARCH  0.4060  1.1546     0.5520   0.5810
1         1       GARCH  0.4210  1.8802     2.7611   0.0058
2         1         GJR  0.3939  1.8964    -0.1583   0.8742
3         1         HAR  0.3960  1.3104        NaN      NaN
4         1        LGBM  0.3421  1.1248    -6.4970   0.0000
5         1  LGBM_noVIX  0.3671  1.2863    -4.0766   0.0000
6         5         HAR  0.2754  0.9952        NaN      NaN
7         5        LGBM  0.2320  0.7283    -5.2434   0.0000
8         5  LGBM_noVIX  0.2569  0.8883    -2.0971   0.0360
9        22         HAR  0.3127  0.8817        NaN      NaN
10       22        LGBM  0.3164  0.7673     0.1220   0.9029
11       22  LGBM_noVIX  0.3258  0.8102     0.4643   0.6425
   horizon       model   QLIKE     MSE  DM vs HAR  p-value
0        1         HAR  0.3416  1.1488        NaN      NaN
1        1        LGBM  0.2968  0.9444    -8.0151   0.0000
2        1  LGBM_noVIX  0.3020  1.1134    -

Hors échantillon, LightGBM avec VIX présente une QLIKE inférieure à celle de LightGBM sans VIX aux horizons de 1, 5 et 22 jours pour le SPX. Pour le TSX, la QLIKE diminue à 1 et 5 jours et augmente à 22 jours. Les p-valeurs affichées proviennent de tests de Diebold–Mariano comparant chaque variante au modèle HAR. La significativité de l’écart entre les deux variantes LightGBM reste à établir par un test direct sur leurs pertes QLIKE appariées.